# MedImageFlow I/O quickstart

This notebook introduces medical image I/O with MedImageFlow. You will read or create a NIfTI image, inspect spatial metadata, optionally inspect a DICOM series, visualize slices, and save a NIfTI copy.

## Learning objectives

- Import the public I/O helpers from `medimageflow.io`.
- Locate example data from any notebook working directory.
- Read NIfTI data and inspect its affine-derived spacing.
- Read a DICOM series when one is available and inspect SimpleITK metadata.
- Visualize slices and MIP views without assuming real patient data.

## Requirements

Install the notebook dependencies from the repository root:

```bash
python -m pip install -e ".[notebooks]"
```

The DICOM section is optional. Place an anonymized DICOM series under `examples/data/dicom_series/` to run it.

## Example data

The notebook uses `examples/data/example_image.nii.gz` when it exists. If it is missing, the notebook creates a synthetic NIfTI image under `examples/data/generated/`. See `examples/data/README.md` before adding external data.

## 1. Imports

In [ ]:
from pathlib import Path

import numpy as np

from medimageflow import mip_visualization, visualization
from medimageflow.io import read_dicom_series, read_nifti, write_nifti

## 2. Configuration

The helper below searches upward for `pyproject.toml`, so the notebook works when launched from the repository root or from `examples/notebooks/`.

In [ ]:
def find_repo_root(start=None):
    start = Path.cwd() if start is None else Path(start)
    for path in (start, *start.parents):
        if (path / "pyproject.toml").exists() and (path / "src" / "medimageflow").exists():
            return path
    raise RuntimeError("Could not find the MedImageFlow repository root.")


REPO_ROOT = find_repo_root()
DATA_DIR = REPO_ROOT / "examples" / "data"
NIFTI_PATH = DATA_DIR / "example_image.nii.gz"
DICOM_DIR = DATA_DIR / "dicom_series"
GENERATED_DIR = DATA_DIR / "generated"

print(REPO_ROOT)
print(DATA_DIR)

## 3. Load or create data

A synthetic NIfTI keeps the first part of the tutorial runnable without downloading data. Real DICOM remains optional because a valid DICOM series cannot be safely synthesized with complete clinical metadata in a short tutorial.

In [ ]:
def create_synthetic_volume() -> tuple[np.ndarray, np.ndarray]:
    z, y, x = np.indices((24, 64, 64))
    sphere = ((z - 12) ** 2 / 36) + ((y - 32) ** 2 / 256) + ((x - 32) ** 2 / 256)
    volume = np.exp(-sphere).astype(np.float32)
    affine = np.diag([1.5, 1.5, 3.0, 1.0]).astype(np.float64)
    return volume, affine


if NIFTI_PATH.exists():
    image_path = NIFTI_PATH
else:
    print("Example NIfTI not found. Creating synthetic data under examples/data/generated/.")
    GENERATED_DIR.mkdir(parents=True, exist_ok=True)
    image_path = GENERATED_DIR / "synthetic_image.nii.gz"
    synthetic_volume, synthetic_affine = create_synthetic_volume()
    write_nifti(synthetic_volume, synthetic_affine, image_path)

volume, affine = read_nifti(image_path)
print(image_path)
print(volume.shape, volume.dtype)

## 4. Run the MedImageFlow workflow

`read_nifti` returns an array and a 4-by-4 affine. The affine maps voxel coordinates to world coordinates. Its column norms are a useful quick check for voxel spacing when there is no shear.

In [ ]:
spacing_from_affine = np.linalg.norm(affine[:3, :3], axis=0)

print("shape:", volume.shape)
print("spacing from affine:", spacing_from_affine)
print("affine:\n", affine)

## 5. Inspect optional DICOM results

This cell runs only when `examples/data/dicom_series/` exists. Missing DICOM data is reported as a clear message instead of stopping the rest of the notebook.

In [ ]:
if DICOM_DIR.exists():
    dicom_image = read_dicom_series(DICOM_DIR)
    print("DICOM size:", dicom_image.GetSize())
    print("DICOM spacing:", dicom_image.GetSpacing())
    print("DICOM origin:", dicom_image.GetOrigin())
    print("DICOM direction:", dicom_image.GetDirection())

    try:
        import pydicom

        dicom_files = [path for path in sorted(DICOM_DIR.rglob("*")) if path.is_file()]
        if dicom_files:
            dataset = pydicom.dcmread(str(dicom_files[0]), stop_before_pixels=True)
            metadata_names = [
                "PatientID",
                "Modality",
                "StudyDate",
                "SeriesDescription",
                "SeriesInstanceUID",
            ]
            for name in metadata_names:
                print(f"{name}:", getattr(dataset, name, "<missing>"))
    except ImportError:
        print("pydicom is not installed; skipping DICOM metadata inspection.")
else:
    print(
        "No DICOM series found. Add anonymized data to "
        "examples/data/dicom_series/ to run this section."
    )

## 6. Visualize results

`visualization` shows central slices of a 3D volume. `mip_visualization` shows maximum-intensity projections along the three spatial axes.

In [ ]:
figure, axes = visualization(volume, spacing=tuple(spacing_from_affine), show=False)
figure

In [ ]:
figure, axes = mip_visualization(volume, spacing=tuple(spacing_from_affine), show=False)
figure

## 7. Save a NIfTI copy

In [ ]:
GENERATED_DIR.mkdir(parents=True, exist_ok=True)
copy_path = write_nifti(volume, affine, GENERATED_DIR / "nifti_copy.nii.gz")
print(copy_path)

## 8. Next steps

NIfTI commonly exposes voxel-to-world geometry through an affine matrix. SimpleITK-backed DICOM reading exposes size, spacing, origin, and direction separately. Be explicit about axis order when mixing formats: MedImageFlow's dataset readers return arrays only, while the lower-level I/O functions expose spatial metadata.